In [11]:
# ==============================================================================
# CELL 1: STRICT DETERMINISM, IMPORTS, AND CONFIGURATION
# ==============================================================================
import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, recall_score

# 1. STRICT HARDWARE DETERMINISM (MUST BE BEFORE TF IMPORT)
SEED = 44
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# 2. IMPORT TENSORFLOW AFTER ENV VARS ARE SET
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

# Force TF operation determinism
tf.config.experimental.enable_op_determinism()

class Config:
    DATA_PATH = 'dataset_meld/'
    TIMESTAMP = '20260718_1827'  # Target NPZ timestamp
    SR = 16000
    MAX_DURATION = 3
    MAX_SAMPLES = SR * MAX_DURATION
    N_MFCC = 40
    MAX_MFCC_LEN = 94 
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 50
    DROPOUT = 0.3
    DEFAULT_CLASSES = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise']


In [12]:
# ==============================================================================
# CELL 2: LOAD FIXED NPZ, MODEL BUILDER, AND RUN BASELINE
# ==============================================================================

# --- 1. DATA LOADING ---
def load_and_prepare_npz(split, timestamp=Config.TIMESTAMP):
    path = os.path.join(Config.DATA_PATH, f"all_features_{split}_{timestamp}.npz")
    data = np.load(path, allow_pickle=True)
    X_mfcc = data['mfcc'].reshape(-1, Config.MAX_MFCC_LEN, Config.N_MFCC)
    X_stats = data['stats']
    meta_cols = data['meta_cols'].tolist()
    y_raw = data['meta'][:, meta_cols.index('emotion')]
    X_text_pred_raw = data['meta'][:, meta_cols.index('textual_sentiment_predicted')]
    return X_mfcc, X_stats, X_text_pred_raw, y_raw

X_train_mfcc, X_train_stats, X_train_txt_raw, y_train_raw = load_and_prepare_npz('train')
X_dev_mfcc, X_dev_stats, X_dev_txt_raw, y_dev_raw = load_and_prepare_npz('dev')
X_test_mfcc, X_test_stats, X_test_txt_raw, y_test_raw = load_and_prepare_npz('test')

# Encode Targets
label_encoder = LabelEncoder().fit(Config.DEFAULT_CLASSES)
y_train = label_encoder.transform(y_train_raw)
y_dev = label_encoder.transform(y_dev_raw)
y_test = label_encoder.transform(y_test_raw)
num_classes = len(Config.DEFAULT_CLASSES)

# Encode Text Modality
sentiment_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_txt = sentiment_encoder.fit_transform(X_train_txt_raw.reshape(-1, 1))
X_dev_txt = sentiment_encoder.transform(X_dev_txt_raw.reshape(-1, 1))
X_test_txt = sentiment_encoder.transform(X_test_txt_raw.reshape(-1, 1))

# Normalize Stats
scaler = StandardScaler().fit(X_train_stats)
X_train_stats = scaler.transform(X_train_stats)
X_dev_stats = scaler.transform(X_dev_stats)
X_test_stats = scaler.transform(X_test_stats)

print(f"✅ Data Encoded. Train Size: {X_train_mfcc.shape[0]}")

# --- 2. MODEL BUILDER ---
def build_hybrid_cnn_lstm(num_classes, stats_dim, sent_dim):
    in_mfcc = layers.Input(shape=(Config.MAX_MFCC_LEN, Config.N_MFCC), name="in_mfcc")
    x_mfcc = layers.Conv1D(64, 3, activation='relu', padding='same')(in_mfcc)
    x_mfcc = layers.MaxPooling1D(2)(x_mfcc)
    x_mfcc = layers.LSTM(64)(x_mfcc)

    in_stats = layers.Input(shape=(stats_dim,), name="in_stats")
    x_stats = layers.Dense(32, activation='relu')(in_stats)
    x_stats = layers.BatchNormalization()(x_stats)

    in_text = layers.Input(shape=(sent_dim,), name="in_text")
    x_text = layers.Dense(16, activation='relu')(in_text)

    merged = layers.Concatenate()([x_mfcc, x_stats, x_text])
    x = layers.Dense(128, activation='relu')(merged)
    x = layers.Dropout(Config.DROPOUT)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=[in_mfcc, in_stats, in_text], outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=Config.LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# --- 3. EVALUATION FORMATTER ---
def evaluate_and_format(model, X_test, y_test, model_name=""):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    
    print(f"\n================ Evaluating {model_name} ================")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, digits=4))
    
    acc = accuracy_score(y_test, y_pred)
    correct_samples = np.sum(y_test == y_pred)
    total_samples = len(y_test)
    print(f"Overall Accuracy: {acc:.4f} ({correct_samples}/{total_samples} samples)\n")
    
    print("--- Per-Class Performance (Accuracy/Recall) ---")
    cm = confusion_matrix(y_test, y_pred)
    for i, cls_name in enumerate(label_encoder.classes_):
        cls_total = np.sum(cm[i, :])
        if cls_total == 0: continue
        cls_correct = cm[i, i]
        cls_acc = cls_correct / cls_total
        print(f"{cls_name.rjust(10)}: {cls_acc:.4f} ({cls_correct}/{cls_total})")

# --- 4. STRICT TRAINING WRAPPER ---
def run_training_strict(X_tr, y_tr, X_v, y_v, X_te, y_te, model_name=""):
    tf.keras.backend.clear_session()
    gc.collect()

    # FORCE RNG RESET RIGHT BEFORE BUILDING THE MODEL
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    model = build_hybrid_cnn_lstm(num_classes, X_tr[1].shape[1], X_tr[2].shape[1])
    
    print(f"\n--- Training {model_name} ---")
    model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
              epochs=Config.EPOCHS, batch_size=Config.BATCH_SIZE, verbose=0)
    
    evaluate_and_format(model, X_te, y_te, model_name)
    return model

✅ Data Encoded. Train Size: 9988


In [13]:
# --- 5. RUN BASELINE ---
X_tr_base = [X_train_mfcc, X_train_stats, X_train_txt]
X_v_base = [X_dev_mfcc, X_dev_stats, X_dev_txt]
X_te_base = [X_test_mfcc, X_test_stats, X_test_txt]

baseline_model = run_training_strict(X_tr_base, y_train, X_v_base, y_dev, X_te_base, y_test, f"Baseline CNN-LSTM (Seed {SEED})")


--- Training Baseline CNN-LSTM (Seed 44) ---

================ Evaluating Baseline CNN-LSTM (Seed 44) ================
              precision    recall  f1-score   support

       anger     0.3058    0.1826    0.2287       345
     disgust     0.1667    0.0147    0.0270        68
        fear     0.0000    0.0000    0.0000        50
         joy     0.4282    0.4453    0.4366       402
     neutral     0.5598    0.8240    0.6667      1256
     sadness     0.2000    0.0577    0.0896       208
    surprise     0.1029    0.0249    0.0401       281

    accuracy                         0.4969      2610
   macro avg     0.2519    0.2213    0.2127      2610
weighted avg     0.4071    0.4969    0.4304      2610

Overall Accuracy: 0.4969 (1297/2610 samples)

--- Per-Class Performance (Accuracy/Recall) ---
     anger: 0.1826 (63/345)
   disgust: 0.0147 (1/68)
      fear: 0.0000 (0/50)
       joy: 0.4453 (179/402)
   neutral: 0.8240 (1035/1256)
   sadness: 0.0577 (12/208)
  surprise: 0.0249 (7

In [ ]:
# ==============================================================================
# CELL 3: AUGMENTATION EXPERIMENT 
# ==============================================================================
def prepare_and_encode(clean_noise, trim_silence, augmentation):
    label_encoder = LabelEncoder().fit(Config.DEFAULT_CLASSES)
    sent_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    splits_out = {}
    skip_report = {}
    for split in ['train', 'dev', 'test']:
        m, s, txt, y, skips = extract_features_for_config(
            SPLIT_DFS[split], clean_noise=clean_noise, trim_silence=trim_silence,
            augmentation=augmentation, is_train=(split == 'train')
        )
        splits_out[split] = {'mfcc': m, 'stats': s, 'sent_raw': txt, 'y_raw': y}
        skip_report[split] = skips

    scaler = StandardScaler().fit(splits_out['train']['stats'])
    sent_encoder.fit(splits_out['train']['sent_raw'].reshape(-1, 1))

    for split in splits_out:
        splits_out[split]['stats'] = scaler.transform(splits_out[split]['stats'])
        splits_out[split]['sent'] = sent_encoder.transform(splits_out[split]['sent_raw'].reshape(-1, 1))
        splits_out[split]['y'] = label_encoder.transform(splits_out[split]['y_raw'])

    return splits_out, skip_report

print("\n--- Running Augmentation: Normal ---")
# Reset seeds before extraction to ensure identical noise patterns
np.random.seed(SEED)
random.seed(SEED)

splits_normal, _ = prepare_and_encode(clean_noise=True, trim_silence=False, augmentation='normal')

X_tr_aug = [splits_normal['train']['mfcc'], splits_normal['train']['stats'], splits_normal['train']['sent']]
X_v_aug = [splits_normal['dev']['mfcc'], splits_normal['dev']['stats'], splits_normal['dev']['sent']]
X_te_aug = [splits_normal['test']['mfcc'], splits_normal['test']['stats'], splits_normal['test']['sent']]

run_training_strict(X_tr_aug, splits_normal['train']['y'], 
                    X_v_aug, splits_normal['dev']['y'], 
                    X_te_aug, splits_normal['test']['y'], 
                    "Augmented CNN-LSTM (Normal)")

In [ ]:
# Repeat for Targeted Augmentation...
print("\n--- Running Augmentation: Targeted ---")
np.random.seed(SEED)
random.seed(SEED)
splits_target, _ = prepare_and_encode(clean_noise=True, trim_silence=False, augmentation='targeted')

X_tr_aug = [splits_target['train']['mfcc'], splits_target['train']['stats'], splits_target['train']['sent']]
X_v_aug = [splits_target['dev']['mfcc'], splits_target['dev']['stats'], splits_target['dev']['sent']]
X_te_aug = [splits_target['test']['mfcc'], splits_target['test']['stats'], splits_target['test']['sent']]

run_training_strict(X_tr_aug, splits_target['train']['y'], 
                    X_v_aug, splits_target['dev']['y'], 
                    X_te_aug, splits_target['test']['y'], 
                    "Augmented CNN-LSTM (Targeted)")